[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Testing and Packaging](https://johnfisher-ai.github.io/Python-Visual-Guides/testing-and-packaging.html)

# Requirements and Pinning


## What you will be able to do

Record what an environment holds with `pip freeze`, rebuild it somewhere else from a requirements
file, and read and write version specifiers such as `==`, `>=` and `~=`. Tell a package you asked for
from one it brought with it, pin both, keep test tools in a requirements file of their own, and read
pip's error when two requirements cannot both be met.


## The idea

### The problem

The **Virtual Environments** notebook made an environment for an old tutorial, with `packaging` 21.3,
and deleted it at the end, as an environment should be. On another computer, or on yours in six
months, the folder is gone, and what survives is whatever was written down.

Written down too loosely, it rebuilds something else. `pip install packaging` installs the newest
release on the day it runs, so a project that recorded only the names of its packages gets a
different set on every install, and one day a release that breaks it, which is how the tutorial
broke. Written down only for the packages you asked for, it still drifts, since `packaging` 21.3
needs `pyparsing`, and nothing said which `pyparsing`. The question is what to record, and how
precisely.

### What a requirements file is

> A **requirements file** lists what `pip install -r` installs, one **requirement** to a line. A
> requirement names a package and can add a **version specifier**, which says which versions are
> acceptable: `==21.3` only that version, `>=2.2` that version or a later one, `~=2.2` a compatible
> release, 2.2 or later within 2, and several of them joined by commas. A requirement that allows
> exactly one version is **pinned**. `pip freeze` prints every package in an environment, pinned,
> including the **transitive dependencies**, the packages that the packages you asked for need. A
> **constraints file** has the same form and installs nothing by itself: it sets the versions of
> whatever a requirements file brings in.

### Why it works that way

- **pip installs the newest version a requirement allows.** A bare name allows every version, so the
  result depends on the day pip runs.
- **A pin gives the same version every time.** `packaging==21.3` installs 21.3 today and next year,
  on any computer that can install it.
- **Packages have packages.** pip chooses the versions of the transitive dependencies too, and
  without a pin it chooses the newest, so a file that pins only what you asked for still changes.
- **`pip freeze` records all of it.** Its output pins every package in the environment, so
  `pip freeze > requirements.txt` on one computer and `pip install -r requirements.txt` on another
  rebuild the same set.
- **A library states ranges, and an application pins.** A library that other projects install says
  which versions it works with, such as `packaging>=22`, so that it fits beside other libraries. A
  program you run, a script, a service or a notebook, pins everything, so that it runs the same way
  every time.
- **A version is not text.** As text, `"2.10"` comes before `"2.9"`. As versions, 2.10 is later, and
  pip compares versions the way the Python packaging specification defines, `packaging` being the
  library that implements it.
- **pip checks all the requirements together.** When no set of versions satisfies every requirement,
  pip says which requirements disagree, and installs nothing.

### Where you will meet this

This library's own repository has a `requirements.txt`, which the workflow that runs its notebooks
installs, and which pins every library a notebook imports, pytest among them. pip's documentation
describes requirements files and constraints files, and the Python packaging specification defines
the version specifiers, `~=` included. A **lock file** records more than a frozen requirements file,
such as where every package came from and a hash of its file. PEP 751 defines a standard one,
`pylock.toml`, accepted in March 2025, and pip has had an experimental `pip lock` command since pip
25.1. The **Project Layout** notebook records a project's own dependencies in `pyproject.toml`, the
other file the **Environments and pip** notebook said this guide would cover.

### What this notebook covers

- `pip freeze`, and a requirements file written from it
- An environment rebuilt from a requirements file, and checked against the original
- A package you asked for, and a package that came with it
- Version specifiers, tried against a list of releases
- Versions compared as versions, not as text
- A constraints file, for the versions of packages you did not ask for
- A requirements file for development, which includes the main one
- A project's environment recorded, rebuilt in another folder, and tested there
- Four errors: a version that does not exist, two requirements that cannot both be met, a
  requirements file pip cannot find, and a freeze of the wrong environment

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
from packaging.specifiers import SpecifierSet
from packaging.version import Version

released = ["2.1", "2.2", "2.2.1", "2.9", "2.10", "3.0"]

for requirement in ["packaging", "packaging>=2.2", "packaging~=2.2", "packaging==2.2.1"]:
    allowed = list(SpecifierSet(requirement.removeprefix("packaging")).filter(released))
    print(f"{requirement:<17} allows {allowed}, so pip installs {max(allowed, key=Version)}")
```

```
packaging         allows ['2.1', '2.2', '2.2.1', '2.9', '2.10', '3.0'], so pip installs 3.0
packaging>=2.2    allows ['2.2', '2.2.1', '2.9', '2.10', '3.0'], so pip installs 3.0
packaging~=2.2    allows ['2.2', '2.2.1', '2.9', '2.10'], so pip installs 2.10
packaging==2.2.1  allows ['2.2.1'], so pip installs 2.2.1
```

Four ways to write one requirement, checked against six releases with the library pip itself uses to
compare versions. A bare name and `>=` install the newest release, which on another day is a
different one. `~=2.2` stays within 2, and `==` allows one version, the same on every day and every
computer.


## Setup

Eight imports, and the two functions the **Virtual Environments** notebook used to run commands.

- `subprocess` runs `venv` and pip as programs of their own
- `sys` names the notebook's own Python, which makes every environment and runs pip
- `os` sets `NO_COLOR`, `PYTHONDONTWRITEBYTECODE` and `COLUMNS` for those programs
- `re` takes the time a test run took out of pytest's report, in `run`
- `SpecifierSet`, from `packaging.specifiers`, checks which versions a specifier allows
- `Version`, from `packaging.version`, compares versions the way pip does
- `Path` names the project's folder, and writes and reads the requirements files
- `shutil` copies the project, and removes the scratch folder at the end

`run` runs a command in the project's folder, `scratch/stations`, and returns its exit code and what
it printed, with the folder's full path and the time a test run took taken out. `pip` runs the
notebook's pip for the Python in an environment, as the **Virtual Environments** notebook explained,
and adds `--no-color`: pip colors its error messages whenever `FORCE_COLOR` is set, as recent
versions of the notebook's kernel set it, whatever `NO_COLOR` says, and this notebook prints some of
those messages. Every environment here is made with `--without-pip`, for the reason that notebook
gave.


In [1]:
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path

from packaging.specifiers import SpecifierSet
from packaging.version import Version

PROJECT = Path("scratch/stations")
(PROJECT / "tests").mkdir(parents=True, exist_ok=True)
os.environ["NO_COLOR"] = "1"                  # programs started from here print without color codes
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"   # and keep no compiled copies, which a quick rewrite can outrun
os.environ["COLUMNS"] = "80"                  # and print reports 80 characters wide


def run(*command, folder=PROJECT):
    """Run a command in a folder, and return its exit code and what it printed, less this computer's paths."""
    finished = subprocess.run([str(part) for part in command], cwd=folder, capture_output=True, text=True)
    printed = (finished.stdout + finished.stderr).replace(f"{Path(folder).resolve()}/", "")
    return finished.returncode, re.sub(r" in \d+\.\d+s\b", "", printed).rstrip()


def pip(environment, *arguments, folder=PROJECT):
    """Run this notebook's pip for the Python in an environment: python -m pip --python ENVIRONMENT ..."""
    return run(sys.executable, "-m", "pip", "--python", environment, "--disable-pip-version-check", "--no-color",
               *arguments, folder=folder)


print("ready:", PROJECT)


ready: scratch/stations


## Worked examples

### pip freeze, and a requirements file written from it

The environment from the **Virtual Environments** notebook, made again: `packaging` 21.3 and
`pyparsing`, which that version needs. `pip freeze` lists what the environment holds, one pinned
requirement to a line, which is exactly the form of a requirements file:


In [2]:
run(sys.executable, "-m", "venv", "--without-pip", "tutorial-env")
code, printed = pip("tutorial-env", "install", "-q", "packaging==21.3", "pyparsing==3.3.2")
print("install exit code:", code)

code, frozen = pip("tutorial-env", "freeze")
(PROJECT / "requirements.txt").write_text(frozen + "\n")

print((PROJECT / "requirements.txt").read_text(), end="")


install exit code: 0
packaging==21.3
pyparsing==3.3.2


In a terminal, with the environment active, `pip freeze > requirements.txt` writes the same file.
The file is the part to keep, in the project's repository, and the environment is the part to throw
away.

### An environment rebuilt from requirements.txt

A new, empty environment, and `pip install -r requirements.txt`, which installs every requirement
in the file. `--dry-run` shows what pip would install without installing it, and then the install
itself runs:


In [3]:
run(sys.executable, "-m", "venv", "--without-pip", "rebuilt")

code, printed = pip("rebuilt", "install", "--dry-run", "-r", "requirements.txt")
print(next(line for line in printed.splitlines() if line.startswith("Would install")))

code, printed = pip("rebuilt", "install", "-q", "-r", "requirements.txt")
print("install exit code:", code)
print("the same packages as tutorial-env:", pip("rebuilt", "freeze")[1] == frozen)


Would install packaging-21.3 pyparsing-3.3.2
install exit code: 0
the same packages as tutorial-env: True


The rebuilt environment holds exactly what the first one held, and the only thing that traveled
between them was a file of two lines. pip's other lines, left out here, report where it found each
package, which depends on what an earlier install left in pip's cache.

### A package you asked for, and a package that came with it

A requirements file written by hand usually lists what the project imports, and nothing else. Here is
one that asks for `packaging` 21.3 alone:


In [4]:
(PROJECT / "direct.txt").write_text("packaging==21.3\n")
run(sys.executable, "-m", "venv", "--without-pip", "direct-env")
pip("direct-env", "install", "-q", "-r", "direct.txt")

code, printed = pip("direct-env", "freeze")
print("asked for:        ", [line.split("==")[0] for line in (PROJECT / "direct.txt").read_text().splitlines()])
print("installed:        ", [line.split("==")[0] for line in printed.splitlines()])

code, printed = pip("direct-env", "show", "packaging", "pyparsing")
print("\n".join(line.strip() for line in printed.splitlines() if line.startswith(("Name", "Requires", "Required-by"))))


asked for:         ['packaging']
installed:         ['packaging', 'pyparsing']
Name: packaging
Requires: pyparsing
Required-by:
Name: pyparsing
Requires:
Required-by: packaging


`pyparsing` came with `packaging`, which requires it, as `pip show` reports. It is a **transitive
dependency**: the project never imports it, and needs it all the same. The file said nothing about
its version, so pip installed the newest `pyparsing` that `packaging` 21.3 accepts, which today is
the version `requirements.txt` pinned, and after the next release of `pyparsing` is a different one.
The cell prints names and not versions for that reason: its output would change with the day it ran.

### Version specifiers

A specifier says which versions a requirement allows. `SpecifierSet`, from `packaging`, the library
pip uses to compare versions, filters a list of releases the way pip would:


In [5]:
released = ["2.1", "2.2", "2.2.1", "2.9", "2.10", "3.0rc1", "3.0"]

for specifier in ["==2.2", "!=2.9", ">=2.2", ">=2.2,<3", "~=2.2", "~=2.2.0", "==2.*"]:
    print(f"{specifier:<9} {list(SpecifierSet(specifier).filter(released))}")


==2.2     ['2.2']
!=2.9     ['2.1', '2.2', '2.2.1', '2.10', '3.0']
>=2.2     ['2.2', '2.2.1', '2.9', '2.10', '3.0']
>=2.2,<3  ['2.2', '2.2.1', '2.9', '2.10']
~=2.2     ['2.2', '2.2.1', '2.9', '2.10']
~=2.2.0   ['2.2', '2.2.1']
==2.*     ['2.1', '2.2', '2.2.1', '2.9', '2.10']


| Specifier | Allows | Here |
|---|---|---|
| `==2.2` | that version and nothing else | `2.2` |
| `!=2.9` | every version but that one | all but `2.9` |
| `>=2.2` | that version or a later one | `2.2` to `3.0` |
| `>=2.2,<3` | both at once: a comma means and | `2.2` to `2.10` |
| `~=2.2` | a compatible release: 2.2 or later, within 2 | `2.2` to `2.10` |
| `~=2.2.0` | 2.2.0 or later, within 2.2 | `2.2` and `2.2.1` |
| `==2.*` | every version whose first number is 2 | `2.1` to `2.10` |

`3.0rc1` appears in no list: a pre-release is left out unless a specifier names one, or pip is told
to accept them. `~=` is shorthand the specification defines, `~=2.2` meaning `>=2.2, ==2.*`, and the
number of parts decides how far it reaches, which is why `~=2.2.0` stops at 2.2.

### Versions compared as versions, not as text

`"2.10"` and `"2.9"` compared as text are compared a character at a time, and `"1"` comes before
`"9"`. `Version` compares them as versions:


In [6]:
print("as text:    ", sorted(released))
print("as versions:", sorted(released, key=Version))
print('"2.10" > "2.9":', "2.10" > "2.9", "| Version:", Version("2.10") > Version("2.9"))
print('2.2 and 2.2.0 are the same version:', Version("2.2") == Version("2.2.0"))


as text:     ['2.1', '2.10', '2.2', '2.2.1', '2.9', '3.0', '3.0rc1']
as versions: ['2.1', '2.2', '2.2.1', '2.9', '2.10', '3.0rc1', '3.0']
"2.10" > "2.9": False | Version: True
2.2 and 2.2.0 are the same version: True


As versions, 2.10 comes after 2.9, a release candidate comes before its release, and 2.2 and 2.2.0
are equal. A program that compares versions compares `Version` objects, never strings, and pip does
the same when it reads a specifier.

### A constraints file

`direct.txt` should keep asking only for what the project imports, and `pyparsing` still needs a
version. A constraints file sets that version without adding `pyparsing` to the requirements: `-c`
limits what `-r` brings in, and a package in the constraints file that nothing requires is not
installed:


In [7]:
(PROJECT / "constraints.txt").write_text("pyparsing==3.2.0\npytest==8.4.2\n")
run(sys.executable, "-m", "venv", "--without-pip", "constrained-env")

code, printed = pip("constrained-env", "install", "--dry-run", "-r", "direct.txt", "-c", "constraints.txt")
print(next(line for line in printed.splitlines() if line.startswith("Would install")))


Would install packaging-21.3 pyparsing-3.2.0


`pyparsing` 3.2.0, as the constraint said, and no pytest, which the constraints file names and
nothing requires. A project with many requirements files can share one constraints file, so that
every file installs the same versions of whatever they have in common.

### A requirements file for development

The tests need pytest, and the program does not, so a project often keeps a second file for the
tools of development. `-r requirements.txt` on its first line includes the main file, so installing
the development file installs both:


In [8]:
(PROJECT / "requirements-dev.txt").write_text("-r requirements.txt\npytest==8.4.2\n")
run(sys.executable, "-m", "venv", "--without-pip", "dev-env")

code, printed = pip("dev-env", "install", "--dry-run", "-r", "requirements-dev.txt")
would_install = next(line for line in printed.splitlines() if line.startswith("Would install"))
names = [item.rsplit("-", 1)[0] for item in would_install.split()[2:]]
print("would install:", sorted(names, key=str.lower))


would install: ['iniconfig', 'packaging', 'pluggy', 'Pygments', 'pyparsing', 'pytest']


The two packages of `requirements.txt`, pytest, and three packages pytest needs: `iniconfig`,
`pluggy` and `Pygments`. pytest needs `packaging` too, and the 21.3 that `requirements.txt` pins
satisfies it. The cell prints names only, since the versions of pytest's own dependencies were not
pinned here, and change with the day.

### A project recorded, rebuilt in another folder, and tested there

The pieces of this notebook, in one project. Its environment gets the development file's tools, `pip
freeze` records every package in it, pinned, and a copy of the project in another folder, standing in
for another computer, rebuilds the environment from that record and runs the tests:


In [9]:
%%writefile scratch/stations/readings.py
"""Readings from the weather stations, and each station's mean temperature."""

import statistics


def mean(values):
    """The mean of the readings that are not None, or None when there are none."""
    present = [value for value in values if value is not None]
    return statistics.fmean(present) if present else None


Writing scratch/stations/readings.py


In [10]:
%%writefile scratch/stations/tests/test_readings.py
from readings import mean


def test_mean_of_two_readings():
    assert mean([4.2, 5.8]) == 5.0


def test_mean_of_no_readings_is_none():
    assert mean([None, None]) is None


Writing scratch/stations/tests/test_readings.py


In [11]:
(PROJECT / "requirements-dev.txt").write_text("pytest==8.4.2\n")
run(sys.executable, "-m", "venv", "--without-pip", ".venv")
pip(".venv", "install", "-q", "-r", "requirements-dev.txt")

code, frozen = pip(".venv", "freeze")
(PROJECT / "requirements-frozen.txt").write_text(frozen + "\n")
print("frozen:", [line.split("==")[0] for line in frozen.splitlines()])

elsewhere = PROJECT.parent / "elsewhere"
shutil.rmtree(elsewhere, ignore_errors=True)
shutil.copytree(PROJECT, elsewhere, ignore=shutil.ignore_patterns("*env", ".venv", "rebuilt", "*.txt", ".pytest_cache"))
shutil.copy(PROJECT / "requirements-frozen.txt", elsewhere)
print("copied:", sorted(str(path.relative_to(elsewhere)) for path in elsewhere.rglob("*") if path.is_file()))

run(sys.executable, "-m", "venv", "--without-pip", ".venv", folder=elsewhere)
pip(".venv", "install", "-q", "-r", "requirements-frozen.txt", folder=elsewhere)
print("the same packages in both places:", pip(".venv", "freeze", folder=elsewhere)[1] == frozen)

code, printed = run(".venv/bin/python", "-m", "pytest", "-q", "--no-header", folder=elsewhere)
print(printed)


frozen: ['iniconfig', 'packaging', 'pluggy', 'Pygments', 'pytest']
copied: ['readings.py', 'requirements-frozen.txt', 'tests/test_readings.py']
the same packages in both places: True
..                                                                       [100%]
2 passed


The copy carried the code, the tests, and one requirements file, and no environment. It rebuilt an
environment with every package at the version the first one had, pytest's own dependencies included,
and the tests passed there. This is how a project reaches a colleague, a server, or the **Continuous
Integration** notebook's runner: as files and a frozen record, never as an environment.

### Where each part came from

| In the project | What it relies on | The section that showed it |
|---|---|---|
| `requirements-frozen.txt` | every package pinned, as `pip freeze` prints them | pip freeze, and a requirements file written from it |
| `pip install -r requirements-frozen.txt` elsewhere | the same set of packages, rebuilt from a file | An environment rebuilt from requirements.txt |
| iniconfig, pluggy and Pygments in the record | a transitive dependency, pinned like any other | A package you asked for, and a package that came with it |
| `pytest==8.4.2` in `requirements-dev.txt` | a specifier that allows one version | Version specifiers |
| the test tools in a file of their own | a development file apart from what the program needs | A requirements file for development |

The frozen file answers the question of the problem: record every package, at exactly the version
that was tested.


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/testing-and-packaging/08-requirements-and-pinning-solutions.ipynb).

**1.** Make an environment called `task-env`, install `packaging==21.3` and `pyparsing==3.3.2` into
it, and write what `pip freeze` lists to `task-requirements.txt` in the project's folder. Print the
file.


In [12]:
# your code here


**2.** Make `task-copy`, install `-r task-requirements.txt` into it, and print whether its `freeze`
is the same as `task-env`'s.


In [13]:
# your code here


**3.** Print which of `["1.0", "1.4", "1.4.5", "1.5", "2.0"]` the specifiers `~=1.4` and `~=1.4.0`
allow.


In [14]:
# your code here


**4.** Sort `["1.10", "1.9", "1.10.post1", "1.10rc1"]` as text, and as versions, and print both.


In [15]:
# your code here


**5.** Write `task-constraints.txt`, which pins `pyparsing` to `3.2.0`, and use `--dry-run` to print
what `packaging==21.3` would install with it.


In [16]:
# your code here


**6.** Write `task-dev.txt`, which includes `task-requirements.txt` and adds `pytest==8.4.2`, and use
`--dry-run` to print the names of the packages it would install.


In [17]:
# your code here


## Common errors

### ERROR: No matching distribution found for packaging==99.0


In [18]:
code, printed = pip("tutorial-env", "install", "packaging==99.0")

print("exit code:", code)
print(printed.splitlines()[-1])


exit code: 1
ERROR: No matching distribution found for packaging==99.0


No release of `packaging` is called 99.0, so no version satisfies the requirement, and pip installed
nothing. The line before this one lists every version pip could find, which grows with every
release, so the cell prints only the last line. A version in a requirements file has to be one that
was released, and `pip index versions packaging` lists them, although pip marks that command
experimental.

### ERROR: Cannot install packaging==21.3 and pyparsing==3.0.5


In [19]:
code, printed = pip("tutorial-env", "install", "packaging==21.3", "pyparsing==3.0.5")

print("exit code:", code)
print("\n".join(line.strip() for line in printed.splitlines() if "depends on" in line or line.startswith("ERROR: Cannot")))


exit code: 1
packaging 21.3 depends on pyparsing!=3.0.5 and >=2.0.2
ERROR: Cannot install packaging==21.3 and pyparsing==3.0.5 because these package versions have conflicting dependencies.


`pyparsing` 3.0.5 exists, and `packaging` 21.3 excludes it: its own requirement is
`pyparsing!=3.0.5 and >=2.0.2`, as the message says. Both requirements can be installed alone, and
not together, so pip installed neither. The message also lists what to try, which differs between
versions of pip, so the cell prints the two lines that name the conflict. Pin a version that both
requirements accept:


In [20]:
code, printed = pip("tutorial-env", "install", "--dry-run", "packaging==21.3", "pyparsing==3.0.4")

print("exit code:", code)
print(next(line for line in printed.splitlines() if line.startswith("Would install")))


exit code: 0
Would install pyparsing-3.0.4


`packaging` 21.3 is already in the environment, so the line lists only what would change:
`pyparsing` 3.0.4, which `packaging` 21.3 accepts.

### ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirement.txt'


In [21]:
code, printed = pip("dev-env", "install", "--dry-run", "-r", "requirement.txt")

print("exit code:", code)
print(printed.splitlines()[-1])


exit code: 1
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirement.txt'


The project's file is `requirements.txt`, and the command asked for `requirement.txt`. pip opens
every requirements file before it looks for a single package, so a name that matches no file stops it
at once, and it installs nothing. The same message appears when the name is right and the command
runs in another folder, since pip looks for the file in the folder it runs in. Check the name, and
the folder:


In [22]:
code, printed = pip("dev-env", "install", "--dry-run", "-r", "requirements.txt")

print("exit code:", code)
print(next(line for line in printed.splitlines() if line.startswith("Would install")))


exit code: 0
Would install packaging-21.3 pyparsing-3.3.2


### No error, and the wrong environment recorded: a freeze of the notebook's own Python


In [23]:
code, printed = run(sys.executable, "-m", "pip", "freeze")
names = [line.split("==")[0].lower() for line in printed.splitlines()]

code, project = pip("tutorial-env", "freeze")
print("ipykernel in the record:", "ipykernel" in names)
print("more packages than tutorial-env holds:", len(names) > len(project.splitlines()))


ipykernel in the record: True
more packages than tutorial-env holds: True


`pip freeze` without `--python` froze the Python the notebook runs in, which holds the notebook's
kernel, `ipykernel`, and everything else installed there, and none of it is what the project needs.
The count differs on every computer, so the cell prints only two facts about it. In a terminal the
same mistake is a `pip freeze` run with no environment active, or with the wrong one. Freeze the
environment the project runs in:


In [24]:
code, printed = pip("tutorial-env", "freeze")

print([line.split("==")[0] for line in printed.splitlines()])


['packaging', 'pyparsing']


Last, the notebook is finished with its files, so this cell removes the scratch folder, with every
environment and the copy of the project:


In [25]:
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


## Recap

- `pip freeze` prints every package in an environment, pinned, and `pip install -r` rebuilds the same
  set from that file on another computer.
- A requirement with no version installs the newest release on the day, and the packages it brings
  come at the newest versions too, unless something pins them.
- `==` pins one version, `>=` and `<` bound a range, `!=` excludes one, `~=2.2` allows 2.2 or later
  within 2, and a comma means both.
- Compare versions with `packaging.version.Version`, never as strings.
- A constraints file sets the versions of whatever a requirements file brings in, and installs
  nothing by itself.
- `-r requirements.txt` at the top of `requirements-dev.txt` adds the tools of development to the
  main requirements.
- When pip reports that versions conflict, read the line that says which package depends on what.


## What is next

The **Project Layout** notebook gives the project a shape that installs: a `src` folder, a
`pyproject.toml` that names its dependencies, and an editable install that lets its tests import it
from any folder.


---

&#8592; **Previous:** [Virtual Environments](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/testing-and-packaging/07-virtual-environments.ipynb)  &nbsp;·&nbsp;  [Testing and Packaging Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/testing-and-packaging.html)
